# Building Your First Agent

In this tutorial, you'll build a complete calculator agent that can perform mathematical operations and tell the current time. This is the foundation for understanding how agents work in NAT.

## What You'll Learn

1. Creating custom function groups (tools)
2. Configuring an LLM provider
3. Building a ReAct agent
4. Testing the workflow
5. Exporting to YAML configuration
6. Running via CLI

## Prerequisites

- NAT SDK installed (see Tutorial 01)
- `NVIDIA_API_KEY` environment variable set
- Basic Python knowledge


In [1]:
import os
import sys
from pathlib import Path

# Setup path for development
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Verify API key
if not os.environ.get("NVIDIA_API_KEY"):
    print("⚠️  Please set NVIDIA_API_KEY environment variable")
else:
    print("✅ Environment configured")


✅ Environment configured


## Step 1: Understanding the Function-Centric Model

From the [Functions documentation](../../../docs/source/workflows/functions/index.md):

> *"Functions (tools) are the main building blocks of NeMo Agent toolkit and define the logic of your workflow."*

Functions are type-safe, asynchronous operations with defined input and output schemas. They provide:
- **Type validation** via Pydantic schemas
- **Composability** through unified interfaces  
- **Asynchronous operation** for better performance

From the [About Workflows documentation](../../../docs/source/workflows/about.md), agents are systems that use LLMs to reason and determine actions. The ReAct agent we'll build uses a **Reasoning + Acting** pattern:

1. **Reason** about the user's request
2. **Act** by calling functions (tools)
3. **Observe** the results
4. **Repeat** until the task is complete


## Step 2: Create the LLM

First, we configure the language model that will power our agent:


In [2]:
from nat.llm.nim_llm import NimLLM

# Create an LLM using NVIDIA NIM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",  # Model to use
    temperature=0.0,                            # Deterministic responses
    max_tokens=1024,                            # Maximum response length
    name="nim_llm",                             # Name for config file
)

print(f"✅ LLM configured: {llm.model_name}")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM configured: meta/llama-3.3-70b-instruct


## Step 3: Create Functions (Tools)

Functions are the building blocks that agents use to perform operations. We'll use two built-in functions:
1. **CurrentTimeTool** - A function that returns the current date and time
2. **CalculatorToolGroup** - A function group providing math operations (add, subtract, multiply, divide)

Function groups package related functions together so they share configuration and resources.


In [3]:
from nat.tool.datetime_tools import CurrentTimeTool

# Create the time tool
time_tool = CurrentTimeTool(name="current_time")

print(f"✅ Time tool created: {time_tool.computed_name}")


✅ Time tool created: current_time


In [4]:
# Import the calculator tool group from the example package
# Note: You need to install it first: uv pip install -e examples/getting_started/simple_calculator
try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    print(f"✅ Calculator tool group created: {calculator.computed_name}")
    HAS_CALCULATOR = True
except ImportError:
    print("⚠️  Calculator not available. Install with:")
    print("   uv pip install -e examples/getting_started/simple_calculator")
    HAS_CALCULATOR = False


✅ Calculator tool group created: calculator


## Step 4: Create the Agent

An agent is itself a function that orchestrates other functions using an LLM. Now we combine the LLM and our functions into a ReAct agent:


In [5]:
from nat.agent.react_agent.register import NatReActAgent

# Create list of tools based on what's available
tools = [time_tool]
if HAS_CALCULATOR:
    tools.append(calculator)

# Create the ReAct agent
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,                          # Show reasoning steps
    parse_agent_response_max_retries=3,    # Retry on parsing errors
)

print(f"✅ Agent created with {len(tools)} tool(s)")


✅ Agent created with 2 tool(s)


## Step 5: Create the Workflow

The NatWorkflow is the container that manages the entire agent lifecycle:


In [6]:
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create the workflow
workflow = NatWorkflow(entrypoint=agent)

print("✅ Workflow created successfully!")


✅ Workflow created successfully!


## Step 6: Test the Workflow

Let's test our agent with some queries:


In [7]:
# Test 1: Simple time query (uncomment to run)
result = await workflow.prompt("What is the current time?")
print("Result:", result)


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Result: 2025-12-07 17:01:50 +0000


In [8]:
# Test 2: Math operation (requires calculator tool)
result = await workflow.prompt("What is 25 * 4?")
print("Result:", result)


Result: 100.0


In [9]:
# Test 3: Combined query (requires both tools)
result = await workflow.prompt("What is 4 * 50 plus the current hour?")
print("Result:", result)


Result: 217.0


## Step 7: Save the Configuration

Export the workflow to a YAML configuration file:


In [10]:
# Create output directory
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

# Save configuration
config_path = config_dir / "calculator_agent.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED YAML CONFIGURATION:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


📄 Configuration saved to: configs/calculator_agent.yaml

GENERATED YAML CONFIGURATION:

functions:
  current_time:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - calculator
  parse_agent_response_max_retries: 3



## Step 8: Run via CLI

You can now run this workflow from the command line:

```bash
# Navigate to the configs directory
cd examples/notebooks/sdk/configs

# Run with a simple query
nat run --config_file calculator_agent.yaml --input "What is 100 divided by 4?"

# Run with a time-based query
nat run --config_file calculator_agent.yaml --input "What time is it?"

# Run with a combined query  
nat run --config_file calculator_agent.yaml --input "What is 4 * 50 plus the current hour?"

# Serve as an API
nat serve --config_file calculator_agent.yaml --port 8000
```


## Understanding the Configuration

Let's break down the generated YAML:

```yaml
# Functions section - individual tools
functions:
  current_time:
    _type: current_datetime  # Built-in tool type

# Function groups - collections of related tools
function_groups:
  calculator:
    _type: calculator        # Provides add, subtract, multiply, divide

# LLM configuration
llms:
  nim_llm:
    _type: nim               # NVIDIA NIM provider
    model: meta/llama-3.3-70b-instruct
    temperature: 0.0
    max_tokens: 1024

# Workflow definition
workflow:
  _type: react_agent         # ReAct agent type
  llm_name: nim_llm          # Reference to LLM
  tool_names:                # List of tools to use
    - current_time
    - calculator
  verbose: true
```

## Key Points

1. **`_type`** - Identifies the component type (registered plugin)
2. **`name`** - How components reference each other
3. **`tool_names`** - List of tools available to the agent
4. **`llm_name`** - Which LLM the agent uses


## Summary

In this tutorial, you learned how to:

✅ Create an LLM configuration using `NimLLM`  
✅ Add tools using `CurrentTimeTool` and custom function groups  
✅ Build a ReAct agent with `NatReActAgent`  
✅ Wrap everything in a `NatWorkflow`  
✅ Export to YAML configuration  
✅ Run via CLI with `nat run`  

## Next Steps

- **[03_agents.ipynb](./03_agents.ipynb)** - Learn about different agent types (ReAct, Tool Calling, ReWOO)
- **[04_functions_and_tools.ipynb](./04_functions_and_tools.ipynb)** - Create custom functions and use MCP/A2A
